# 03: Transformer Introduction

Understand self-attention and use pre-trained Transformer models for inference.

## Topics
- Self-attention mechanism (conceptual and code)
- Using Hugging Face pipelines
- Understanding tokenizer output
- Running inference with pre-trained models

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import pipeline, AutoTokenizer, AutoModel

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 1. Self-Attention: The Core Innovation

Self-attention lets each token in a sequence directly attend to every other token, regardless of distance.

The formula:
```
Attention(Q, K, V) = softmax(QKᵀ / √dₖ) × V
```

- **Q (Query)**: What am I looking for?
- **K (Key)**: What do I contain?
- **V (Value)**: What information do I provide?

In [ ]:
# Simplified self-attention from scratch

def self_attention_simple(Q, K, V):
    """Compute scaled dot-product attention."""
    d_k = Q.shape[-1]
    # Compute attention scores
    scores = torch.matmul(Q, K.transpose(-2, -1))
    # Scale by sqrt(d_k)
    scores = scores / np.sqrt(d_k)
    # Apply softmax to get attention weights
    attention_weights = torch.softmax(scores, dim=-1)
    # Weighted sum of values
    output = torch.matmul(attention_weights, V)
    return output, attention_weights


# Example: 3 tokens, embedding dimension 4
seq_len = 3
d_model = 4

# Random embeddings for 3 tokens
torch.manual_seed(42)
tokens = torch.randn(seq_len, d_model)

# Q, K, V are projections of the input (in real Transformers, learned)
Q = tokens  # Query
K = tokens  # Key
V = tokens  # Value

output, attention_weights = self_attention_simple(Q, K, V)

print("Input tokens shape:", tokens.shape)
print("Attention weights shape:", attention_weights.shape)
print("\nAttention weights (who attends to whom):")
print(attention_weights.numpy().round(3))
print("\nNote: Each row sums to 1.0 (probability distribution)")

In [ ]:
# Visualize attention weights
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Attention weights heatmap
im = axes[0].imshow(attention_weights.detach().numpy(), cmap='Blues')
axes[0].set_xticks(range(seq_len))
axes[0].set_yticks(range(seq_len))
axes[0].set_xticklabels([f'Token {i}' for i in range(seq_len)])
axes[0].set_yticklabels([f'Token {i}' for i in range(seq_len)])
axes[0].set_xlabel('Key (attended to)')
axes[0].set_ylabel('Query (attending from)')
axes[0].set_title('Self-Attention Weights')

for i in range(seq_len):
    for j in range(seq_len):
        axes[0].text(j, i, f'{attention_weights[i, j]:.2f}', ha='center', va='center')

plt.colorbar(im, ax=axes[0])

# Before/after comparison
axes[1].imshow(tokens.detach().numpy(), cmap='RdYlBu', aspect='auto')
axes[1].set_xlabel('Embedding Dimension')
axes[1].set_ylabel('Token Index')
axes[1].set_title('Input Token Embeddings')
axes[1].set_yticks(range(seq_len))
axes[1].set_yticklabels([f'Token {i}' for i in range(seq_len)])

plt.tight_layout()
plt.show()

## 2. Using Hugging Face Pipelines

Hugging Face `pipeline` provides a high-level API for common NLP tasks. It handles tokenization, model loading, and post-processing.

In [ ]:
# Sentiment Analysis
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

texts = [
    "I absolutely loved this movie, it was fantastic!",
    "This is the worst product I have ever bought.",
    "The weather is okay today, nothing special.",
    "I'm not sure how I feel about this.",
    "Absolutely brilliant work! Highly recommend.",
]

results = sentiment_analyzer(texts)

print("Sentiment Analysis Results:")
print("=" * 60)
for text, result in zip(texts, results):
    emoji = "pos" if result["label"] == "POSITIVE" else "neg"
    print(f"[{emoji}] {result["score"]:.4f} | {text[:50]}")

## 3. Understanding the Tokenizer

Tokenizers convert text into token IDs that models understand. Modern models use subword tokenization.

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

sample_text = "Transformers have revolutionized natural language processing!"

# Tokenize
encoded = tokenizer(sample_text, return_tensors="pt")

print("Original text:")
print(f"  {sample_text}")
print(f"\nToken IDs: {encoded['input_ids'].tolist()}")
print(f"Attention mask: {encoded['attention_mask'].tolist()}")

# Decode tokens back to text
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
print(f"\nTokens: {tokens}")
print(f"\nNumber of tokens: {len(tokens)}")

In [ ]:
# Subword tokenization examples
print("Subword Tokenization Examples:")
print("=" * 50)

test_words = ["transformers", "unhappiness", "人工智能", "running"]

for word in test_words:
    tokens = tokenizer.tokenize(word)
    ids = tokenizer.encode(word)
    print(f"\n'{word}'")
    print(f"  Tokens: {tokens}")
    print(f"  IDs: {ids}")

## 4. Running Inference with Pre-trained Models

Let's explore different types of pre-trained models and their outputs.

In [ ]:
# Named Entity Recognition (NER)
ner_pipeline = pipeline("ner", aggregation_strategy="simple")

ner_text = "Elon Musk founded SpaceX in Hawthorne, California in 2002."
ner_results = ner_pipeline(ner_text)

print("Named Entity Recognition:")
print(f"Input: {ner_text}")
print(f"\nEntities found:")
for entity in ner_results:
    print(f"  [{entity["entity_group"]}] {entity["word"]:20s} (score: {entity["score"]:.4f})")

In [ ]:
# Question Answering
qa_pipeline = pipeline("question-answering")

context = """
The Transformer architecture was introduced in the paper "Attention Is All You Need" 
by Vaswani et al. in 2017. It replaced recurrent neural networks (RNNs) and long 
short-term memory (LSTM) networks for many NLP tasks. The key innovation was the 
self-attention mechanism, which allows the model to process all tokens in parallel.
"""

questions = [
    "Who introduced the Transformer?",
    "When was it published?",
    "What replaced RNNs?",
    "What is the key innovation?",
]

print("Question Answering:")
print("=" * 60)
for question in questions:
    result = qa_pipeline(question=question, context=context)
    print(f"Q: {question}")
    print(f"A: {result['answer']}")
    print(f"   Score: {result['score']:.4f}")
    print()

In [ ]:
# Text Generation
generator = pipeline(
    "text-generation",
    model="gpt2",
    max_length=60,
    num_return_sequences=2,
    do_sample=True,
    temperature=0.7
)

prompts = [
    "The future of artificial intelligence is",
    "In a world where machines can think",
]

print("Text Generation (GPT-2):")
print("=" * 60)
for prompt in prompts:
    results = generator(prompt)
    print(f"\nPrompt: '{prompt}'")
    for i, result in enumerate(results):
        print(f"\n  Generated {i+1}:")
        print(f"  {result['generated_text'][:200]}")

In [ ]:
# Summarization
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

article = """
Artificial intelligence has made remarkable progress in recent years. Large language 
models like GPT-4 and Claude can now write code, analyze data, and engage in 
sophisticated reasoning. These models are trained on vast amounts of text data and 
can be fine-tuned for specific tasks. However, they also raise important questions 
about bias, privacy, and the future of work. Companies are racing to deploy AI 
systems while governments worldwide are developing regulations to ensure safe and 
responsible use of this powerful technology.
"""

summary = summarizer(article, max_length=60, min_length=20)

print("Summarization (BART):")
print("=" * 60)
print(f"\nOriginal ({len(article.split())} words):")
print(article.strip())
print(f"\nSummary ({len(summary[0]['summary_text'].split())} words):")
print(summary[0]['summary_text'])

## 5. Comparing Different Models

Different pre-trained models specialize in different tasks.

In [ ]:
# Compare sentiment analysis models
models_to_compare = [
    "distilbert-base-uncased-finetuned-sst-2-english",
    "nlptown/bert-base-multilingual-uncased-sentiment",
]

test_text = "The movie was absolutely wonderful and I loved every minute of it!"

print("Model Comparison - Sentiment Analysis")
print("=" * 70)
print(f"Text: '{test_text}'")
print()

for model_name in models_to_compare:
    analyzer = pipeline("sentiment-analysis", model=model_name)
    result = analyzer(test_text)[0]
    print(f"Model: {model_name}")
    print(f"  Label: {result['label']}, Score: {result['score']:.4f}")
    print()

## Key Takeaways

1. **Self-attention** lets each token directly attend to all other tokens — no sequential dependency
2. **Hugging Face pipelines** make it easy to use pre-trained models for common tasks
3. **Tokenizers** convert text to model-readable token IDs using subword tokenization
4. **Pre-trained models** can be used directly for inference (sentiment, NER, QA, generation, summarization)
5. **Different models** specialize in different tasks — choose the right one for your use case